<h1 align="center"><b>Validate June Sign Up Files</b></h1>

**1. Install Required Packages**

In [17]:
# Package used to connect to MySQL Databases
import mysql.connector
import pymysql
import paramiko
from sqlalchemy import create_engine

#Connect To SFTP
import pysftp

# Folder Creation
import os
import glob
from pathlib import Path
from dotenv import load_dotenv

# Data Manipulation Packages
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Data Visualisation
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Package To Ignore Warnings
import warnings
warnings.filterwarnings("ignore")


**2. Grab Credentials From Enviromental Variable**

In [18]:
ROOT_DIR: Path = Path().resolve().parent
load_dotenv(os.path.join(ROOT_DIR, ".env"))
Root = os.path.normpath(os.getcwd() + os.sep + os.pardir)

In [19]:
host_ = os.getenv('HOST')
database_ = os.getenv('DATABASE')
user_ = os.getenv('NAME')
password_ = os.getenv('PASSWORD')
port_ = os.getenv('PORT')

**3. Create MySQL Script**

**GHANA**

In [20]:
GH_Q = """WITH affiliate_tx AS 
                        (SELECT *
                                ,RANK() OVER (PARTITION BY profile_id ORDER BY created DESC) AS Ranking
                        FROM betika_bi_gh.affiliate_tx
                        WHERE DATE(created) BETWEEN '2025-06-01' AND '2025-06-30' 
                        AND transaction_type IN ('REGISTRATION')
                        AND affiliate_tag IS NOT NULL AND affiliate_tag NOT IN ('None', 'Null','Default','')
                        ),
        Affiliates AS

                (SELECT  DATE(a.created) AS ACCOUNT_OPENING_DATE 
                        ,a.affiliate_tag AS BTAG
                        ,a.profile_id
                        ,b.MSISDN AS USERNAME
                        ,'GH' AS PLAYER_COUNTRY
                FROM affiliate_tx AS a

                LEFT JOIN betika_bi_gh.profile AS b
                ON a.profile_id = b.profile_id

                WHERE a.Ranking = 1
                ),


        KPI AS 
                (SELECT profile_id
                        ,summary_date
                        ,dep AS Deposit

                FROM betika_bi_gh.f_kpi_gh
                
                WHERE profile_id IN (SELECT profile_id FROM Affiliates) 
                ),  

        Deposits AS
                (SELECT profile_id
                ,first_deposit AS FIRST_DEPOSIT_DATE
                FROM betika_bi_gh.dim_first_last_gh
                WHERE profile_id IN (SELECT profile_id FROM Affiliates)
                ) 

        SELECT CONCAT(233,a.profile_id) AS PLAYER_ID
        ,a.USERNAME
        ,a.BTAG
        ,a.PLAYER_COUNTRY
        ,a.ACCOUNT_OPENING_DATE
        ,b.FIRST_DEPOSIT_DATE
        ,c.DEPOSIT AS FIRST_DEPOSIT_AMOUNT
        ,CASE WHEN c.DEPOSIT > 0 THEN 1 END AS FTD
        FROM Affiliates AS a

        LEFT JOIN Deposits AS b
        ON a.profile_id = b.profile_id

        LEFT JOIN KPI AS c
        ON b.profile_id = c.profile_id
        AND b.FIRST_DEPOSIT_DATE = c.summary_date;"""

**DRC**

In [21]:
CD_Q = """WITH affiliate_tx AS 
                        (SELECT *
                                ,RANK() OVER (PARTITION BY profile_id ORDER BY created DESC) AS Ranking
                        FROM betika_bi_cd.affiliate_tx
                        WHERE DATE(created) BETWEEN '2025-06-01' AND '2025-06-30' 
                        AND transaction_type IN ('REGISTRATION')
                        AND affiliate_tag IS NOT NULL AND affiliate_tag NOT IN ('None', 'Null','Default','')
                        ),
        Affiliates AS

                (SELECT  DATE(a.created) AS ACCOUNT_OPENING_DATE 
                        ,a.affiliate_tag AS BTAG
                        ,a.profile_id
                        ,b.MSISDN AS USERNAME
                        ,'CD' AS PLAYER_COUNTRY
                FROM affiliate_tx AS a

                LEFT JOIN betika_bi_cd.profile AS b
                ON a.profile_id = b.profile_id

                WHERE a.Ranking = 1
                ),


        KPI AS 
                (SELECT profile_id
                        ,summary_date
                        ,dep AS Deposit

                FROM betika_bi_cd.f_kpi_drc
                
                WHERE profile_id IN (SELECT profile_id FROM Affiliates) 
                ),  

        Deposits AS
                (SELECT profile_id
                ,first_deposit AS FIRST_DEPOSIT_DATE
                FROM betika_bi_cd.dim_first_last_drc
                WHERE profile_id IN (SELECT profile_id FROM Affiliates)
                ) 

        SELECT CONCAT(243,a.profile_id) AS PLAYER_ID 
        ,a.USERNAME
        ,a.BTAG
        ,a.PLAYER_COUNTRY
        ,a.ACCOUNT_OPENING_DATE
        ,b.FIRST_DEPOSIT_DATE
        ,c.DEPOSIT AS FIRST_DEPOSIT_AMOUNT
        ,CASE WHEN c.DEPOSIT > 0 THEN 1 END AS FTD
        FROM Affiliates AS a

        LEFT JOIN Deposits AS b
        ON a.profile_id = b.profile_id

        LEFT JOIN KPI AS c
        ON b.profile_id = c.profile_id
        AND b.FIRST_DEPOSIT_DATE = c.summary_date;"""

**UGANDA**

In [22]:
UG_Q = """WITH affiliate_tx AS 
                        (SELECT *
                                ,RANK() OVER (PARTITION BY profile_id ORDER BY created DESC) AS Ranking
                        FROM betika_bi_ug.affiliate_tx
                        WHERE DATE(created) BETWEEN '2025-06-01' AND '2025-06-30' 
                        AND transaction_type IN ('REGISTRATION')
                        AND affiliate_tag IS NOT NULL AND affiliate_tag NOT IN ('None', 'Null','Default','')
                        ),
        Affiliates AS

                (SELECT  DATE(a.created) AS ACCOUNT_OPENING_DATE 
                        ,a.affiliate_tag AS BTAG
                        ,a.profile_id
                        ,b.MSISDN AS USERNAME
                        ,'UG' AS PLAYER_COUNTRY
                FROM affiliate_tx AS a

                LEFT JOIN betika_bi_ug.profile AS b
                ON a.profile_id = b.profile_id

                ),


        KPI AS 
                (SELECT profile_id
                        ,summary_date
                        ,dep AS Deposit

                FROM betika_bi_ug.f_kpi_ug
                
                WHERE profile_id IN (SELECT profile_id FROM Affiliates) 
                ),  

        Deposits AS
                (SELECT profile_id
                ,first_deposit AS FIRST_DEPOSIT_DATE
                FROM betika_bi_ug.dim_first_last_ug
                WHERE profile_id IN (SELECT profile_id FROM Affiliates)
                ) 

        SELECT CONCAT(256,a.profile_id) AS PLAYER_ID
        ,a.USERNAME
        ,a.BTAG
        ,a.PLAYER_COUNTRY
        ,a.ACCOUNT_OPENING_DATE
        ,b.FIRST_DEPOSIT_DATE
        ,c.DEPOSIT AS FIRST_DEPOSIT_AMOUNT
        ,CASE WHEN c.DEPOSIT > 0 THEN 1 END AS FTD
        FROM Affiliates AS a

        LEFT JOIN Deposits AS b
        ON a.profile_id = b.profile_id

        LEFT JOIN KPI AS c
        ON b.profile_id = c.profile_id
        AND b.FIRST_DEPOSIT_DATE = c.summary_date;"""

**4. Run MySQL Script & Return Result As A DataFrame**

In [23]:
# Code To Connect MySQL
cobi_betika = mysql.connector.connect(host=host_
                                      ,database=database_
                                      ,user=user_
                                      ,password=password_
                                      ,port=port_)

# Connect to MySQL database
try:
    with cobi_betika.cursor() as cursor:
        GH = pd.read_sql(GH_Q,cobi_betika)
        CD = pd.read_sql(CD_Q,cobi_betika)
        UG = pd.read_sql(UG_Q,cobi_betika)

finally:
    cobi_betika.close()

In [24]:
SU = pd.concat([GH, CD, UG])
SU['FIRST_DEPOSIT_AMOUNT'] = SU['FIRST_DEPOSIT_AMOUNT'].abs()
SU['SIGN_UPS'] = 1

**5. Count The Number Of Sign Ups From The Database**

In [25]:
dbSignUps = SU.groupby(['ACCOUNT_OPENING_DATE'])[['SIGN_UPS']].sum().reset_index().sort_values(by=['ACCOUNT_OPENING_DATE'])
dbSignUps['ACCOUNT_OPENING_DATE'] = dbSignUps['ACCOUNT_OPENING_DATE'].astype(str)

**6. Import Every Sign Up File For The Month Of June**

In [26]:
folder_path = '/Users/katlegomatebane/Documents/Github/bi-etl/Broken Files/Validate June Numbers/Sign Up Files/'

csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

dataframes = {}

for file_path in csv_files:
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    dataframes[file_name] = pd.read_csv(file_path)

**7. Exclude Data From Ethiopia From The Sign Up Files**

In [27]:
for name, df in dataframes.items():
    dataframes[name] = df[df['PLAYER_COUNTRY'] != 'ET']

**8. Count The Number Of SignUps In Each Sign Up File**

In [28]:
row_counts = {name: len(df) for name, df in dataframes.items()}

**9. Save The Number Of Sign Ups For Each Sign Up File As A DataFrame**

In [29]:
signUps = pd.DataFrame(list(row_counts.items()), columns=['FILE NAME', 'SignUps'])
signUps = signUps.sort_values(by=['FILE NAME'])
signUps = signUps.reset_index(inplace=False,drop=True)
signUps['ACCOUNT_OPENING_DATE']= signUps['FILE NAME'].str[-10:]

**10. Compare Data From MySQL With Data From All The Sign Up Files**

In [30]:
df = pd.merge(signUps, dbSignUps, on='ACCOUNT_OPENING_DATE')
df.rename(columns={'SignUps': 'FROM INDIVIDUAL SALES FILES',
                   'SIGN_UPS': 'FROM MySQL DATABSE',
                   'ACCOUNT_OPENING_DATE':'ACCOUNT OPENING DATE'}, inplace=True)

df['MATCH'] = df['FROM INDIVIDUAL SALES FILES'] == df['FROM MySQL DATABSE']
#df[['FILE NAME','ACCOUNT OPENING DATE','FROM INDIVIDUAL SALES FILES','FROM MySQL DATABSE','MATCH']]

**11. Import Excel File With MyAffiliates Data**

In [31]:
MA = pd.read_excel('MyAffiliates - Advanced Reporting - Affiliate Report (15).xlsx')
MA = MA[['Date','Signups']]
MA = MA.rename(columns={'Signups':'FROM MYAFFILIATES','Date':'ACCOUNT OPENING DATE'})
MA['ACCOUNT OPENING DATE'] = MA['ACCOUNT OPENING DATE'].astype(str)

**12. Merge All Data & Compare Results**

In [32]:
All = pd.merge(df,MA, on='ACCOUNT OPENING DATE')
All['AFFILIATES MATCH'] = All['FROM INDIVIDUAL SALES FILES'] == All['FROM MYAFFILIATES']
All['DIFFRENCE'] = All['FROM INDIVIDUAL SALES FILES'] - All['FROM MYAFFILIATES']
All[['FILE NAME','ACCOUNT OPENING DATE','MATCH','FROM INDIVIDUAL SALES FILES','FROM MySQL DATABSE','FROM MYAFFILIATES','AFFILIATES MATCH','DIFFRENCE']]

,FILE NAME,ACCOUNT OPENING DATE,MATCH,FROM INDIVIDUAL SALES FILES,FROM MySQL DATABSE,FROM MYAFFILIATES,AFFILIATES MATCH,DIFFRENCE
0,betika_signups_2025-06-01,2025-06-01,True,589,589,591,False,-2
1,betika_signups_2025-06-02,2025-06-02,True,370,370,370,True,0
2,betika_signups_2025-06-03,2025-06-03,True,345,345,345,True,0
3,betika_signups_2025-06-04,2025-06-04,True,785,785,785,True,0
4,betika_signups_2025-06-05,2025-06-05,True,846,846,847,False,-1
5,betika_signups_2025-06-06,2025-06-06,True,478,478,479,False,-1
6,betika_signups_2025-06-07,2025-06-07,True,859,859,860,False,-1
7,betika_signups_2025-06-08,2025-06-08,True,912,912,914,False,-2
8,betika_signups_2025-06-09,2025-06-09,True,445,445,445,True,0
9,betika_signups_2025-06-10,2025-06-10,True,896,896,869,False,27


In [33]:
All[['FROM INDIVIDUAL SALES FILES','FROM MySQL DATABSE','FROM MYAFFILIATES','DIFFRENCE']].sum()

FROM INDIVIDUAL SALES FILES    21581
FROM MySQL DATABSE             21581
FROM MYAFFILIATES              21594
DIFFRENCE                        -13
dtype: int64